In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 08 · Registro del modelo

> **Esqueleto.** Guía completa: `curso/kit-estudiante/guia-validacion-registro-y-app.md`, sección 5.

En la etapa de optimización el **tracking** guardó *ejecuciones*: cada ensayo, con sus parámetros y métricas. Eso
responde "¿qué probamos?". El **registro de modelos** responde otra pregunta: **"¿cuál es el modelo que usamos, y en
qué versión?"**

| | Tracking | Registro |
|---|---|---|
| Guarda | Ejecuciones (muchas, de prueba) | Versiones del modelo (pocas, candidatas a producción) |
| Pregunta | ¿Qué probamos y qué dio? | ¿Qué modelo está en uso y qué versión? |
| Ejemplo | 30 ensayos de la búsqueda | `mi_modelo` versión 2, alias `champion` |

Un **alias** (`champion`) es una etiqueta móvil: quien use el modelo pide "el champion" y no le importa qué número de
versión es. Si mañana hay uno mejor, se mueve el alias y nadie toca el código.

## 0. Conexión con el seguimiento

In [ ]:
try:
    from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment
except ModuleNotFoundError:
    configure_mlflow = None
    print("TODO: el módulo src/proyecto/tracking/mlflow_utils.py viene en la plantilla; si no está, vuelve a clonar la rama base.")

if configure_mlflow is None:
    donde = experimento = None
else:
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        donde = configure_mlflow()
    for aviso in avisos:
        print("Aviso:", aviso.message)
    experimento = set_experiment("mi-proyecto-modelo-final")   # TODO: cambia "mi-proyecto" por el nombre de tu proyecto
    print("MLflow:", donde, "| experimento:", experimento)

> **Si ves `local`:** no hay conexión con Databricks (sin `.env`, sin red o cuota agotada). El registro funciona igual,
> pero en tu equipo (`mlflow_local.db`). Es el respaldo previsto.

## 1. Cargar el modelo final y los resultados de la prueba

Salen del notebook 07: el modelo en `models/final_pipeline.joblib` y las métricas en `reports/07_metricas_test.json`.

In [ ]:
import json

import joblib

raiz = Path.cwd().resolve().parent
modelo_path, info_path = raiz / "models" / "final_pipeline.joblib", raiz / "reports" / "07_metricas_test.json"

modelo = joblib.load(modelo_path) if modelo_path.exists() else None
info = json.loads(info_path.read_text(encoding="utf-8")) if info_path.exists() else None
if modelo is None or info is None:
    print("Pendiente: termina el notebook 07 (guarda final_pipeline.joblib y 07_metricas_test.json).")
else:
    print("modelo y resultados cargados")

## 2. Qué se registra

El modelo final **con lo que hace falta para confiar en él**:

- los **parámetros** y el **tipo de modelo**,
- las **métricas en la prueba** (la evaluación que se hizo una sola vez),
- una **firma** (qué columnas espera y de qué tipo) y un **ejemplo de entrada**,
- **etiquetas** que dicen de dónde salió.

> **Nota técnica.** MLflow guarda por defecto los modelos de scikit-learn en un formato que **rechaza** un `Pipeline` con
> preprocesamiento ("untrusted types"). Por eso el registro usa `serialization_format="cloudpickle"`. Los archivos
> *pickle* **pueden ejecutar código al cargarse**: carga solo modelos que guardaste tú o tu grupo, nunca uno de una
> fuente en la que no confíes.

In [ ]:
import mlflow

NOMBRE_MODELO = "mi_modelo"          # TODO: nombre corto con guiones bajos, sin espacios
COLUMNAS_ENTRADA = None              # TODO: lista de las columnas CRUDAS que una persona conoce al decidir (sin la columna objetivo)

listo_para_registrar = all(x is not None for x in (modelo, info, donde, COLUMNAS_ENTRADA)) and NOMBRE_MODELO != "mi_modelo"
if not listo_para_registrar:
    print("Pendiente: carga el modelo (sección 1), la conexión (sección 0) y define NOMBRE_MODELO y COLUMNAS_ENTRADA.")
else:
    nombre_completo = registered_model_name(NOMBRE_MODELO)
    ejemplo = joblib.load(raiz / "data" / "processed" / "train.joblib")[COLUMNAS_ENTRADA].head(5)
    with mlflow.start_run(run_name="modelo-final") as run:
        mlflow.log_params({"modelo": str(info["decisiones"]["modelo_final"])})   # TODO: añade los parámetros que quieras registrar
        # TODO: mlflow.log_metrics({...}) con las métricas de la prueba (números, no textos)
        mlflow.set_tags({"etapa": "08_registro", "nota": "prueba abierta una sola vez en la etapa 7"})
        mlflow.sklearn.log_model(
            modelo, name="model", input_example=ejemplo, registered_model_name=nombre_completo,
            serialization_format="cloudpickle",   # ver la nota técnica de arriba: sin esto MLflow rechaza un Pipeline completo
        )
        run_id = run.info.run_id
    print("ejecución:", run_id, "| modelo registrado como:", nombre_completo)

## 3. Poner el alias `champion`

Busca la versión recién registrada y dale el alias. A partir de ahora cualquiera pide `models:/<nombre>@champion`.

In [ ]:
if listo_para_registrar:
    cliente = mlflow.MlflowClient()
    version = max(int(v.version) for v in cliente.search_model_versions(f"name = '{nombre_completo}'"))
    cliente.set_registered_model_alias(nombre_completo, "champion", str(version))
    print(f"{nombre_completo}: versión {version} -> alias 'champion'")
else:
    print("Pendiente: registra el modelo (sección 2).")

## 4. Comprobación de ida y vuelta

Un registro que no se puede **recuperar** no sirve. Carga el modelo desde el registro y comprueba que predice
**exactamente igual** que el modelo local.

In [ ]:
if listo_para_registrar:
    recuperado = mlflow.sklearn.load_model(f"models:/{nombre_completo}@champion")
    muestra = joblib.load(raiz / "data" / "processed" / "train.joblib")[COLUMNAS_ENTRADA].head(200)
    import numpy as np
    iguales = np.allclose(modelo.predict_proba(muestra), recuperado.predict_proba(muestra))
    assert iguales, "el modelo recuperado del registro no coincide con el original"
    print("el modelo recuperado del registro predice igual que el original:", iguales)
else:
    print("Pendiente: registra el modelo y asigna el alias.")

## 5. Guardar la constancia

Anota dónde quedó el modelo en `reports/08_registro.json` y completa `reports/08_registro.md`: qué versión es la
`champion`, dónde está registrada (Databricks o local) y quién la registró.

In [ ]:
if listo_para_registrar:
    constancia = {"tracking": donde, "experimento": experimento, "modelo_registrado": nombre_completo,
                  "version": version, "alias": "champion", "run_id": run_id}
    (raiz / "reports" / "08_registro.json").write_text(json.dumps(constancia, indent=2, ensure_ascii=False), encoding="utf-8")
    print("guardado: reports/08_registro.json")
else:
    print("Pendiente.")

## Qué sigue

El notebook 09 exporta el modelo `champion` a un archivo (`models/model.joblib`) que viaja con la app. **La app no
consulta Databricks**: no se ponen credenciales en la nube, y el despliegue no depende de la cuota de la plataforma.
El registro sigue siendo la **fuente de verdad**: el archivo siempre se exporta desde la versión `champion`.